# SDF Parser

## Configuration

In [1]:
#pull in local configuration
%run config.py
!cat config.py

# config.py
# This file wants to be listed in .gitignore

GNINA_LOC = "/home/dwaine/octoberproject/gnina"
GNINA_PARAMETER = "--no_gpu"


In [44]:
sdf = "sourcefiles/sm3ps.sdf"
#sdf = "sourcefiles/test.sdf"
outputdir = "outputfiles/"

## Parse SDF into temporary Dataframe

In [39]:
import pandas as pd
from rdkit.Chem import PandasTools

sdf_path = sdf

decoydf = PandasTools.LoadSDF(
    sdf_path,
    molColName="ROMol",
    smilesName="SMILES",
    includeFingerprints=False,
    removeHs=False,
    strictParsing=True
)

# Add numbered ID column
decoydf["ID"] = [f"decoy{i+1}" for i in range(len(decoydf))]

print(decoydf.head())
print(decoydf.columns)
print(decoydf.shape)     # (rows, columns)
print(decoydf.info(memory_usage='deep'))

  CERAPP_ID                                              CASRN  \
0     10001  50-00-0|630-08-0|13007-92-6|13463-39-3|13463-4...   
1     10002                         50-02-2|378-44-9|1249-18-9   
2     10003                                            50-03-3   
3     10004                                            50-04-4   
4     10005                                    50-06-6|57-30-7   

                                       CHEMICAL NAME  \
0  Formaldehyde|Carbon monoxide|Chromium carbonyl...   
1  Dexamethasone|Betamethasone|(11beta)-9-fluoro-...   
2                             Hydrocortisone acetate   
3                                  Cortisone acetate   
4                 Phenobarbital|Phenobarbital sodium   

                                     Original_SMILES  \
0  C=O|[O+]#[C-]|[O+]#[C-].[O+]#[C-].[O+]#[C-].[O...   
1  [C@]12([C@](C(=O)CO)(O)[C@H](C)C[C@@]1([H])[C@...   
2  [C@]12([C@](C(COC(=O)C)=O)(O)CCC1C1CCC3[C@@](C...   
3  [C@]12([C@](C(COC(=O)C)=O)(O)CC[C@@]1([

In [47]:
outputdf = None
# columns:   Chem_name,	CAS,	consensus_2_agonist,	Potency_class_2_agonist,	consensus_2_antagonist,	Potency_class_2_antagonist

inputdf = decoydf.head(5)
inputdf = decoydf

for idx, row in inputdf.iterrows():
    #These four rows may have multiple values separated by "|" symbol.
    casrn = row["CASRN"]
    chemical_name = row["CHEMICAL NAME"]
    smiles = row["Original_SMILES"]
    source = row["SOURCE"]

    c2ag = row["consensus_2_agonist"]
    pc2ag = row["Potency_class_2_agonist"]
    c2an = row["consensus_2_antagonist"]
    pc2an = row["Potency_class_2_antagonist"]
    
    #print("One record: ")
    #print(idx, casrn, smiles, source)


    if "|" in casrn and "|" in chemical_name and "|" in smiles and "|" in source:
        casrn_parts = casrn.split("|")
        chemical_name_parts = chemical_name.split("|")
        smiles_parts = smiles.split("|")
        source_parts = source.split("|")
        
        for index, value in enumerate(casrn_parts):
            
            #print("Subrecord:")
            #print(index, casrn_parts[index], chemical_name_parts[index], smiles_parts[index], source_parts[index])           
            
            # new row as dictionary
            new_row = {
                "Chem_name": chemical_name_parts[index], 
                "CAS": casrn_parts[index],
                "consensus_2_agonist": c2ag,
                "Potency_class_2_agonist": pc2ag,
                "consensus_2_antagonist": c2an,
                "Potency_class_2_antagonist": pc2an
            }

            # append to end
            outputdf = pd.concat([outputdf, pd.DataFrame([new_row])], ignore_index=True)

    else:
        # new row as dictionary
        new_row = {
                "Chem_name": chemical_name, 
                "CAS": casrn,
                "consensus_2_agonist": c2ag,
                "Potency_class_2_agonist": pc2ag,
                "consensus_2_antagonist": c2an,
                "Potency_class_2_antagonist": pc2an
        }

        # append to end
        outputdf = pd.concat([outputdf, pd.DataFrame([new_row])], ignore_index=True)
        
            
print(outputdf.head(20))    

                                            Chem_name         CAS  \
0                                        Formaldehyde     50-00-0   
1                                     Carbon monoxide    630-08-0   
2                                   Chromium carbonyl  13007-92-6   
3                                     Nickel carbonyl  13463-39-3   
4                 Iron carbonyl (Fe(CO)5), (TB-5-11)-  13463-40-6   
5                                    Paraformaldehyde  30525-89-4   
6                                octacarbonyldicobalt  10210-68-1   
7                                     Nickel carbonyl  12612-55-4   
8   tri-?-carbonyltetracarbonyl(pentacarbonyldicob...  50696-78-1   
9                                       Dexamethasone     50-02-2   
10                                      Betamethasone    378-44-9   
11  (11beta)-9-fluoro-11,17,21-trihydroxy-16-methy...   1249-18-9   
12                             Hydrocortisone acetate     50-03-3   
13                                

In [49]:
# Write to filesystem

outputdf.to_csv(outputdir + "cerapp.csv", index=False)